# Email analysis code

In [2]:
import mailbox
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
import os

os.chdir('/workspaces/emails_analysis')

client = OpenAI()

mbox = mailbox.mbox("Flights.mbox")

results = []

def extract_flight_info(email_text):
    prompt = f"""
    Extract the following fields from this flight email:
    - flight_date
    - departure_airport
    - arrival_airport
    - destination_city
    - flight_number

    Return ONLY a JSON object.

    Email:
    {email_text[:4000]}
    """

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return response.choices[0].message.content


for message in tqdm(mbox):
    body = ""

    if message.is_multipart():
        for part in message.walk():
            if part.get_content_type() == "text/plain":
                body = part.get_payload(decode=True).decode(errors="ignore")
                break
    else:
        body = message.get_payload(decode=True).decode(errors="ignore")

    try:
        parsed = extract_flight_info(body)
        results.append(parsed)
    except:
        continue

df = pd.DataFrame(results)
df.to_csv("flights.csv", index=False)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable